In [9]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

from longeval.spark import get_spark
from pyspark.sql import functions as F

spark = get_spark(cores=8, memory="20g")

data_root = Path("~/shared/longeval/evaluation/scores").expanduser()
df = spark.read.parquet(data_root.as_posix())
df.printSchema()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


25/05/30 05:53:01 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


root
 |-- map: double (nullable = true)
 |-- ndcg: double (nullable = true)
 |-- ndcg_cut_10: double (nullable = true)
 |-- ndcg_rel: double (nullable = true)
 |-- qid: string (nullable = true)
 |-- experiment: string (nullable = true)
 |-- date: string (nullable = true)



In [12]:
df.groupBy("experiment").agg(
    F.round(F.avg("ndcg_cut_10"), 3).alias("ndcg10_mean"),
    F.round(F.stddev("ndcg_cut_10"), 3).alias("ndcg10_stddev"),
).orderBy("ndcg10_mean", ascending=False).show(100, truncate=False)

+----------------------+-----------+-------------+
|experiment            |ndcg10_mean|ndcg10_stddev|
+----------------------+-----------+-------------+
|bm25-reranked         |0.296      |0.371        |
|bm25-expanded-reranked|0.295      |0.375        |
|bm25                  |0.242      |0.337        |
|bm25-expanded         |0.194      |0.314        |
+----------------------+-----------+-------------+



In [14]:
df.groupBy("date").agg(
    F.round(F.avg("ndcg_cut_10"), 3).alias("ndcg10_mean"),
    F.round(F.stddev("ndcg_cut_10"), 3).alias("ndcg10_stddev"),
).orderBy("date").show(100, truncate=False)

+-------+-----------+-------------+
|date   |ndcg10_mean|ndcg10_stddev|
+-------+-----------+-------------+
|2022-06|0.139      |0.28         |
|2022-07|0.145      |0.287        |
|2022-08|0.146      |0.285        |
|2022-09|0.225      |0.335        |
|2022-10|0.309      |0.365        |
|2022-11|0.306      |0.367        |
|2022-12|0.316      |0.364        |
|2023-01|0.32       |0.367        |
|2023-02|0.324      |0.374        |
|2023-03|0.326      |0.376        |
|2023-04|0.332      |0.372        |
|2023-05|0.336      |0.375        |
|2023-06|0.324      |0.379        |
|2023-07|0.331      |0.372        |
|2023-08|0.291      |0.371        |
+-------+-----------+-------------+



In [20]:
(
    df.groupBy("experiment", "date")
    .agg(
        F.round(F.avg("ndcg_cut_10"), 3).alias("ndcg10_mean"),
    )
    .groupBy(
        "date",
    )
    .pivot("experiment")
    .agg(F.first("ndcg10_mean"))
    .orderBy("date")
).show()

+-------+-----+-------------+----------------------+-------------+
|   date| bm25|bm25-expanded|bm25-expanded-reranked|bm25-reranked|
+-------+-----+-------------+----------------------+-------------+
|2022-06|0.127|         0.11|                 0.157|        0.161|
|2022-07|0.134|        0.116|                 0.163|        0.168|
|2022-08|0.141|        0.123|                  NULL|        0.176|
|2022-09| 0.21|        0.184|                 0.249|        0.259|
|2022-10|0.296|        0.236|                 0.344|        0.361|
|2022-11|0.292|        0.226|                  0.34|        0.367|
|2022-12|0.303|        0.238|                  0.35|        0.372|
|2023-01|0.312|        0.237|                 0.352|        0.379|
|2023-02| 0.31|        0.251|                 0.356|        0.378|
|2023-03|0.316|        0.246|                 0.354|         0.39|
|2023-04|0.323|        0.253|                 0.361|         0.39|
|2023-05|0.327|        0.255|                 0.368|        0.